# End-to-End Pipeline Evaluation

## Purpose
Evaluate the **complete pipeline flow** from CV input through all three stages:
- **Stage 1**: Named Entity Recognition (NER) - Extract skills from CV/JD
- **Stage 2**: Skill Gap Analysis - Identify gaps between CV and JD
- **Stage 3**: Course Recommendations - Suggest courses to fill gaps

## Key Metrics
1. **Pipeline Success Rate**: % of test cases completing all stages without errors
2. **Stage Latency**: Time taken at each stage (p50, p95, p99)
3. **End-to-End Latency**: Total time from CV input to recommendations
4. **Quality Propagation**: How errors in Stage 1 affect Stage 2 and 3 quality
5. **Output Validity**: % of recommendations that are actionable

## Test Scenarios
- ✅ Happy path: Complete CV + JD → recommendations
- ⚠️ Partial CV: Missing sections → graceful degradation
- ❌ Invalid input: Malformed data → error handling
- 🔄 Edge cases: Empty skills, no gaps, no matching courses

**Last Updated**: 2026-04-19

In [0]:
# Standard library
import sys
import time
import json
from pathlib import Path
from typing import Dict, List, Tuple, Any
from dataclasses import dataclass, asdict

# Data processing
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Dependencies loaded")
print(f"📅 Evaluation Date: 2026-04-19")

In [0]:
# Detect repository root dynamically
import os

current_path = os.getcwd()
print(f"Current working directory: {current_path}")

# Navigate up to find repo root (contains 'skillup' directory)
if '/Workspace/Users/' in current_path:
    # Extract user directory
    user_dir = current_path.split('/Workspace/Users/')[1].split('/')[0]
    REPO_ROOT = f"/Workspace/Users/{user_dir}/skillup"
else:
    # Fallback: assume we're already in skillup or subdirectory
    REPO_ROOT = str(Path(current_path).parent.parent if 'evaluation' in current_path else current_path)

print(f"✅ Repository root: {REPO_ROOT}")

# Add to path for imports
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
    print(f"✅ Added {REPO_ROOT} to sys.path")

In [0]:
@dataclass
class PipelineResult:
    """Store results from end-to-end pipeline execution."""
    test_id: str
    success: bool
    
    # Stage 1: NER
    stage1_success: bool
    stage1_latency_ms: float
    cv_skills_extracted: int
    jd_skills_extracted: int
    stage1_error: str = ""
    
    # Stage 2: SkillGap
    stage2_success: bool
    stage2_latency_ms: float
    gaps_identified: int
    gap_priorities: List[float] = None
    stage2_error: str = ""
    
    # Stage 3: Recommender
    stage3_success: bool
    stage3_latency_ms: float
    courses_recommended: int
    avg_relevance_score: float = 0.0
    stage3_error: str = ""
    
    # Overall
    total_latency_ms: float = 0.0
    
    def __post_init__(self):
        if self.gap_priorities is None:
            self.gap_priorities = []
        self.total_latency_ms = self.stage1_latency_ms + self.stage2_latency_ms + self.stage3_latency_ms

print("✅ PipelineResult dataclass defined")

In [0]:
# Load test CVs and JDs
test_data_path = Path(REPO_ROOT) / "data" / "test_profiles"

# Mock test data if files don't exist
if test_data_path.exists():
    # Load from files
    test_cases = []
    for cv_file in test_data_path.glob("cv_*.txt"):
        test_id = cv_file.stem
        jd_file = test_data_path / f"jd_{test_id.split('_')[1]}.txt"
        
        if jd_file.exists():
            test_cases.append({
                'test_id': test_id,
                'cv_text': cv_file.read_text(),
                'jd_text': jd_file.read_text()
            })
    print(f"✅ Loaded {len(test_cases)} test cases from {test_data_path}")
else:
    # Create mock test cases
    test_cases = [
        {
            'test_id': 'test_001',
            'cv_text': """Data Analyst with 3 years experience. 
            Skills: Python, SQL, Excel, Tableau, pandas, data visualization.
            Looking to transition into Machine Learning.""",
            'jd_text': """Machine Learning Engineer needed.
            Required: Python, scikit-learn, TensorFlow, deep learning, model deployment, MLOps.
            Nice to have: PyTorch, Kubernetes, Docker."""
        },
        {
            'test_id': 'test_002',
            'cv_text': """Frontend Developer with React and JavaScript.
            Skills: React, JavaScript, HTML, CSS, Git, REST APIs.""",
            'jd_text': """Full-Stack Developer position.
            Required: React, Node.js, MongoDB, Express.js, Docker, AWS.
            Backend skills essential."""
        },
        {
            'test_id': 'test_003',
            'cv_text': """Senior Software Engineer.
            Skills: Java, Spring Boot, Microservices, Kubernetes, AWS, CI/CD, Agile.""",
            'jd_text': """Principal Engineer for cloud architecture.
            Required: System design, Kubernetes, Terraform, multi-cloud, security, leadership."""
        }
    ]
    print(f"⚠️  Test data path not found. Using {len(test_cases)} mock test cases")

print(f"\n📊 Test Cases Summary:")
for tc in test_cases:
    print(f"  - {tc['test_id']}: CV {len(tc['cv_text'])} chars, JD {len(tc['jd_text'])} chars")

In [0]:
# Attempt to import actual modules with fallback to mocks
try:
    # Stage 1: NER
    from modules.ner import extract_skills_from_text
    print("✅ Imported NER module")
except ImportError:
    print("⚠️  NER module not found, using mock")
    def extract_skills_from_text(text: str) -> List[str]:
        """Mock skill extraction."""
        import re
        # Simple keyword extraction
        common_skills = ['python', 'sql', 'java', 'javascript', 'react', 'aws', 'docker', 
                        'kubernetes', 'tensorflow', 'scikit-learn', 'pandas', 'tableau',
                        'mongodb', 'express', 'node.js', 'spring boot', 'microservices']
        found_skills = [skill for skill in common_skills if skill.lower() in text.lower()]
        return found_skills

try:
    # Stage 2: SkillGap
    from modules.skill_gap import analyze_skill_gap
    print("✅ Imported SkillGap module")
except ImportError:
    print("⚠️  SkillGap module not found, using mock")
    def analyze_skill_gap(cv_skills: List[str], jd_skills: List[str]) -> Dict[str, Any]:
        """Mock skill gap analysis."""
        gaps = [skill for skill in jd_skills if skill not in cv_skills]
        return {
            'gaps': gaps,
            'gap_count': len(gaps),
            'priorities': [0.8 - i*0.1 for i in range(len(gaps))]  # Descending priorities
        }

try:
    # Stage 3: Recommender
    from modules.recommender import recommend_courses
    print("✅ Imported Recommender module")
except ImportError:
    print("⚠️  Recommender module not found, using mock")
    def recommend_courses(gaps: List[str], top_k: int = 5) -> List[Dict[str, Any]]:
        """Mock course recommendations."""
        return [
            {'course_id': f'course_{i+1}', 'skill': gap, 'relevance': 0.9 - i*0.1}
            for i, gap in enumerate(gaps[:top_k])
        ]

In [0]:
results: List[PipelineResult] = []

print("🚀 Running End-to-End Pipeline Evaluation...\n")

for test_case in test_cases:
    test_id = test_case['test_id']
    print(f"\n{'='*60}")
    print(f"Test Case: {test_id}")
    print(f"{'='*60}")
    
    result = PipelineResult(
        test_id=test_id,
        success=False,
        stage1_success=False,
        stage1_latency_ms=0.0,
        cv_skills_extracted=0,
        jd_skills_extracted=0,
        stage2_success=False,
        stage2_latency_ms=0.0,
        gaps_identified=0,
        stage3_success=False,
        stage3_latency_ms=0.0,
        courses_recommended=0
    )
    
    # Stage 1: NER
    try:
        start = time.time()
        cv_skills = extract_skills_from_text(test_case['cv_text'])
        jd_skills = extract_skills_from_text(test_case['jd_text'])
        result.stage1_latency_ms = (time.time() - start) * 1000
        
        result.cv_skills_extracted = len(cv_skills)
        result.jd_skills_extracted = len(jd_skills)
        result.stage1_success = len(cv_skills) > 0 and len(jd_skills) > 0
        
        print(f"✅ Stage 1 (NER): {result.stage1_latency_ms:.1f}ms")
        print(f"   CV Skills: {cv_skills}")
        print(f"   JD Skills: {jd_skills}")
    except Exception as e:
        result.stage1_error = str(e)
        print(f"❌ Stage 1 Failed: {e}")
        results.append(result)
        continue
    
    # Stage 2: SkillGap
    try:
        start = time.time()
        gap_analysis = analyze_skill_gap(cv_skills, jd_skills)
        result.stage2_latency_ms = (time.time() - start) * 1000
        
        result.gaps_identified = gap_analysis.get('gap_count', 0)
        result.gap_priorities = gap_analysis.get('priorities', [])
        result.stage2_success = result.gaps_identified >= 0
        
        print(f"✅ Stage 2 (SkillGap): {result.stage2_latency_ms:.1f}ms")
        print(f"   Gaps Identified: {gap_analysis.get('gaps', [])}")
    except Exception as e:
        result.stage2_error = str(e)
        print(f"❌ Stage 2 Failed: {e}")
        results.append(result)
        continue
    
    # Stage 3: Recommender
    try:
        start = time.time()
        recommendations = recommend_courses(gap_analysis.get('gaps', []), top_k=5)
        result.stage3_latency_ms = (time.time() - start) * 1000
        
        result.courses_recommended = len(recommendations)
        if recommendations:
            result.avg_relevance_score = np.mean([r.get('relevance', 0.0) for r in recommendations])
        result.stage3_success = len(recommendations) > 0
        
        print(f"✅ Stage 3 (Recommender): {result.stage3_latency_ms:.1f}ms")
        print(f"   Courses Recommended: {result.courses_recommended}")
        print(f"   Avg Relevance: {result.avg_relevance_score:.3f}")
    except Exception as e:
        result.stage3_error = str(e)
        print(f"❌ Stage 3 Failed: {e}")
        results.append(result)
        continue
    
    # Mark as successful if all stages passed
    result.success = result.stage1_success and result.stage2_success and result.stage3_success
    print(f"\n{'✅ Pipeline SUCCESS' if result.success else '❌ Pipeline FAILED'}")
    print(f"Total Latency: {result.total_latency_ms:.1f}ms")
    
    results.append(result)

print(f"\n{'='*60}")
print(f"✅ Evaluation Complete: {len(results)} test cases processed")
print(f"{'='*60}")

In [0]:
# Convert results to DataFrame for analysis
results_df = pd.DataFrame([asdict(r) for r in results])

# Calculate aggregate metrics
metrics = {
    'total_tests': len(results),
    'successful_pipelines': results_df['success'].sum(),
    'pipeline_success_rate': results_df['success'].mean() * 100,
    
    # Stage-wise success rates
    'stage1_success_rate': results_df['stage1_success'].mean() * 100,
    'stage2_success_rate': results_df['stage2_success'].mean() * 100,
    'stage3_success_rate': results_df['stage3_success'].mean() * 100,
    
    # Latency statistics (successful runs only)
    'stage1_latency_p50': results_df[results_df['stage1_success']]['stage1_latency_ms'].median(),
    'stage1_latency_p95': results_df[results_df['stage1_success']]['stage1_latency_ms'].quantile(0.95),
    
    'stage2_latency_p50': results_df[results_df['stage2_success']]['stage2_latency_ms'].median(),
    'stage2_latency_p95': results_df[results_df['stage2_success']]['stage2_latency_ms'].quantile(0.95),
    
    'stage3_latency_p50': results_df[results_df['stage3_success']]['stage3_latency_ms'].median(),
    'stage3_latency_p95': results_df[results_df['stage3_success']]['stage3_latency_ms'].quantile(0.95),
    
    'total_latency_p50': results_df[results_df['success']]['total_latency_ms'].median(),
    'total_latency_p95': results_df[results_df['success']]['total_latency_ms'].quantile(0.95),
    
    # Quality metrics
    'avg_skills_extracted_cv': results_df['cv_skills_extracted'].mean(),
    'avg_skills_extracted_jd': results_df['jd_skills_extracted'].mean(),
    'avg_gaps_identified': results_df['gaps_identified'].mean(),
    'avg_courses_recommended': results_df['courses_recommended'].mean(),
    'avg_relevance_score': results_df[results_df['avg_relevance_score'] > 0]['avg_relevance_score'].mean()
}

print("📊 Pipeline Metrics Summary")
print("="*60)
print(f"\n🎯 Success Rates:")
print(f"  Overall Pipeline: {metrics['pipeline_success_rate']:.1f}%")
print(f"  Stage 1 (NER): {metrics['stage1_success_rate']:.1f}%")
print(f"  Stage 2 (SkillGap): {metrics['stage2_success_rate']:.1f}%")
print(f"  Stage 3 (Recommender): {metrics['stage3_success_rate']:.1f}%")

print(f"\n⚡ Latency (ms):")
print(f"  Stage 1 - P50: {metrics['stage1_latency_p50']:.1f}ms, P95: {metrics['stage1_latency_p95']:.1f}ms")
print(f"  Stage 2 - P50: {metrics['stage2_latency_p50']:.1f}ms, P95: {metrics['stage2_latency_p95']:.1f}ms")
print(f"  Stage 3 - P50: {metrics['stage3_latency_p50']:.1f}ms, P95: {metrics['stage3_latency_p95']:.1f}ms")
print(f"  Total   - P50: {metrics['total_latency_p50']:.1f}ms, P95: {metrics['total_latency_p95']:.1f}ms")

print(f"\n📈 Quality Metrics:")
print(f"  Avg CV Skills Extracted: {metrics['avg_skills_extracted_cv']:.1f}")
print(f"  Avg JD Skills Extracted: {metrics['avg_skills_extracted_jd']:.1f}")
print(f"  Avg Gaps Identified: {metrics['avg_gaps_identified']:.1f}")
print(f"  Avg Courses Recommended: {metrics['avg_courses_recommended']:.1f}")
print(f"  Avg Relevance Score: {metrics['avg_relevance_score']:.3f}")

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Stage-wise success rates
stages = ['Stage 1\n(NER)', 'Stage 2\n(SkillGap)', 'Stage 3\n(Recommender)', 'Overall\nPipeline']
success_rates = [
    metrics['stage1_success_rate'],
    metrics['stage2_success_rate'],
    metrics['stage3_success_rate'],
    metrics['pipeline_success_rate']
]

colors = ['#2ecc71' if rate >= 90 else '#f39c12' if rate >= 70 else '#e74c3c' for rate in success_rates]
axes[0].bar(stages, success_rates, color=colors, alpha=0.7, edgecolor='black')
axes[0].axhline(y=90, color='green', linestyle='--', linewidth=1, label='Target: 90%')
axes[0].set_ylabel('Success Rate (%)', fontsize=11)
axes[0].set_title('Stage-wise Success Rates', fontsize=12, fontweight='bold')
axes[0].set_ylim(0, 105)
axes[0].legend()
for i, rate in enumerate(success_rates):
    axes[0].text(i, rate + 2, f'{rate:.1f}%', ha='center', fontweight='bold')

# Plot 2: Test case outcomes
outcomes = results_df['success'].value_counts()
outcome_labels = ['Success' if x else 'Failed' for x in outcomes.index]
outcome_colors = ['#2ecc71' if x else '#e74c3c' for x in outcomes.index]
axes[1].pie(outcomes.values, labels=outcome_labels, autopct='%1.1f%%', 
            colors=outcome_colors, startangle=90, textprops={'fontsize': 11, 'fontweight': 'bold'})
axes[1].set_title('Pipeline Outcome Distribution', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ Success rate visualizations created")

In [0]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Filter successful runs for latency analysis
successful_runs = results_df[results_df['success']]

if len(successful_runs) > 0:
    # Plot 1: Stage-wise latency box plots
    latency_data = [
        successful_runs['stage1_latency_ms'],
        successful_runs['stage2_latency_ms'],
        successful_runs['stage3_latency_ms']
    ]
    axes[0, 0].boxplot(latency_data, labels=['Stage 1\n(NER)', 'Stage 2\n(SkillGap)', 'Stage 3\n(Recommender)'])
    axes[0, 0].set_ylabel('Latency (ms)', fontsize=11)
    axes[0, 0].set_title('Stage-wise Latency Distribution', fontsize=12, fontweight='bold')
    axes[0, 0].grid(axis='y', alpha=0.3)
    
    # Plot 2: Total latency histogram
    axes[0, 1].hist(successful_runs['total_latency_ms'], bins=10, color='#3498db', alpha=0.7, edgecolor='black')
    axes[0, 1].axvline(metrics['total_latency_p50'], color='green', linestyle='--', linewidth=2, label=f"P50: {metrics['total_latency_p50']:.1f}ms")
    axes[0, 1].axvline(metrics['total_latency_p95'], color='red', linestyle='--', linewidth=2, label=f"P95: {metrics['total_latency_p95']:.1f}ms")
    axes[0, 1].set_xlabel('Total Latency (ms)', fontsize=11)
    axes[0, 1].set_ylabel('Frequency', fontsize=11)
    axes[0, 1].set_title('End-to-End Latency Distribution', fontsize=12, fontweight='bold')
    axes[0, 1].legend()
    axes[0, 1].grid(axis='y', alpha=0.3)
    
    # Plot 3: Latency breakdown per test case
    x = np.arange(len(successful_runs))
    width = 0.6
    axes[1, 0].bar(x, successful_runs['stage1_latency_ms'], width, label='Stage 1', color='#e74c3c', alpha=0.8)
    axes[1, 0].bar(x, successful_runs['stage2_latency_ms'], width, 
                   bottom=successful_runs['stage1_latency_ms'], label='Stage 2', color='#f39c12', alpha=0.8)
    axes[1, 0].bar(x, successful_runs['stage3_latency_ms'], width,
                   bottom=successful_runs['stage1_latency_ms'] + successful_runs['stage2_latency_ms'],
                   label='Stage 3', color='#2ecc71', alpha=0.8)
    axes[1, 0].set_xlabel('Test Case', fontsize=11)
    axes[1, 0].set_ylabel('Latency (ms)', fontsize=11)
    axes[1, 0].set_title('Latency Breakdown by Test Case', fontsize=12, fontweight='bold')
    axes[1, 0].set_xticks(x)
    axes[1, 0].set_xticklabels([r['test_id'] for _, r in successful_runs.iterrows()], rotation=45)
    axes[1, 0].legend()
    axes[1, 0].grid(axis='y', alpha=0.3)
    
    # Plot 4: Quality metrics per test case
    x = np.arange(len(results_df))
    axes[1, 1].plot(x, results_df['cv_skills_extracted'], marker='o', label='CV Skills', linewidth=2)
    axes[1, 1].plot(x, results_df['jd_skills_extracted'], marker='s', label='JD Skills', linewidth=2)
    axes[1, 1].plot(x, results_df['gaps_identified'], marker='^', label='Gaps', linewidth=2)
    axes[1, 1].plot(x, results_df['courses_recommended'], marker='d', label='Courses', linewidth=2)
    axes[1, 1].set_xlabel('Test Case', fontsize=11)
    axes[1, 1].set_ylabel('Count', fontsize=11)
    axes[1, 1].set_title('Quality Metrics by Test Case', fontsize=12, fontweight='bold')
    axes[1, 1].set_xticks(x)
    axes[1, 1].set_xticklabels(results_df['test_id'], rotation=45)
    axes[1, 1].legend()
    axes[1, 1].grid(alpha=0.3)
else:
    for ax in axes.flat:
        ax.text(0.5, 0.5, 'No successful runs to visualize', 
                ha='center', va='center', fontsize=12)
        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
plt.show()

print("✅ Latency and quality visualizations created")

In [0]:
print("🔍 Error Propagation Analysis")
print("="*60)

# Analyze how Stage 1 errors affect downstream stages
stage1_failures = results_df[~results_df['stage1_success']]
if len(stage1_failures) > 0:
    print(f"\n⚠️  Stage 1 Failures: {len(stage1_failures)}")
    print(f"   Impact: {len(stage1_failures)} pipelines blocked (100% propagation)")
    for _, row in stage1_failures.iterrows():
        print(f"   - {row['test_id']}: {row['stage1_error']}")
else:
    print("\n✅ No Stage 1 failures detected")

# Analyze Stage 2 errors (given Stage 1 success)
stage2_failures = results_df[results_df['stage1_success'] & ~results_df['stage2_success']]
if len(stage2_failures) > 0:
    print(f"\n⚠️  Stage 2 Failures (after Stage 1 success): {len(stage2_failures)}")
    for _, row in stage2_failures.iterrows():
        print(f"   - {row['test_id']}: {row['stage2_error']}")
else:
    print("\n✅ No Stage 2 failures detected (given Stage 1 success)")

# Analyze Stage 3 errors (given Stage 1 & 2 success)
stage3_failures = results_df[
    results_df['stage1_success'] & 
    results_df['stage2_success'] & 
    ~results_df['stage3_success']
]
if len(stage3_failures) > 0:
    print(f"\n⚠️  Stage 3 Failures (after Stage 1 & 2 success): {len(stage3_failures)}")
    for _, row in stage3_failures.iterrows():
        print(f"   - {row['test_id']}: {row['stage3_error']}")
else:
    print("\n✅ No Stage 3 failures detected (given Stage 1 & 2 success)")

# Quality degradation analysis
print(f"\n📉 Quality Degradation Analysis:")
print(f"   Avg skills extracted: CV={metrics['avg_skills_extracted_cv']:.1f}, JD={metrics['avg_skills_extracted_jd']:.1f}")
if metrics['avg_skills_extracted_cv'] < 3 or metrics['avg_skills_extracted_jd'] < 3:
    print("   ⚠️  Warning: Low skill extraction may indicate NER quality issues")

print(f"   Avg gaps identified: {metrics['avg_gaps_identified']:.1f}")
if metrics['avg_gaps_identified'] < 1:
    print("   ⚠️  Warning: Low gap detection may indicate SkillGap issues")

print(f"   Avg courses recommended: {metrics['avg_courses_recommended']:.1f}")
if metrics['avg_courses_recommended'] < metrics['avg_gaps_identified'] * 0.5:
    print("   ⚠️  Warning: Low recommendation rate may indicate course catalog issues")

In [0]:
print("🎯 Target Threshold Validation")
print("="*60)

# Define targets
targets = {
    'pipeline_success_rate': 90.0,
    'stage1_success_rate': 95.0,
    'stage2_success_rate': 90.0,
    'stage3_success_rate': 85.0,
    'total_latency_p95': 500.0,  # ms
    'avg_relevance_score': 0.70
}

validation_results = []

for metric_name, target_value in targets.items():
    actual_value = metrics.get(metric_name, 0.0)
    
    # Determine if target is met (latency is lower-is-better)
    if 'latency' in metric_name:
        passed = actual_value <= target_value
        comparison = f"{actual_value:.1f} <= {target_value:.1f}"
    else:
        passed = actual_value >= target_value
        comparison = f"{actual_value:.2f} >= {target_value:.2f}"
    
    status = "✅ PASS" if passed else "❌ FAIL"
    
    validation_results.append({
        'metric': metric_name,
        'target': target_value,
        'actual': actual_value,
        'passed': passed
    })
    
    print(f"{status} {metric_name}: {comparison}")

# Overall assessment
passed_count = sum(1 for r in validation_results if r['passed'])
total_count = len(validation_results)
overall_pass_rate = (passed_count / total_count) * 100

print(f"\n{'='*60}")
print(f"Overall: {passed_count}/{total_count} targets met ({overall_pass_rate:.1f}%)")

if overall_pass_rate >= 80:
    print("✅ Pipeline meets quality standards")
elif overall_pass_rate >= 60:
    print("⚠️  Pipeline needs improvement")
else:
    print("❌ Pipeline requires significant work")

In [0]:
# Create export directoryexport_dir = EVAL_ARTIFACTSexport_dir.mkdir(parents=True, exist_ok=True)# Export detailed resultsresults_file = export_dir / "e2e_pipeline_results.csv"results_df.to_csv(results_file, index=False)print(f"✅ Detailed results exported to {results_file}")# Export summary metricssummary = {    'evaluation_date': '2026-04-19',    'total_tests': len(results),    'metrics': metrics,    'validation_results': validation_results,    'targets_met': f"{passed_count}/{total_count}"}summary_file = export_dir / "e2e_pipeline_summary.json"with open(summary_file, 'w') as f:    json.dump(summary, f, indent=2, default=str)print(f"✅ Summary metrics exported to {summary_file}")print(f"\n📊 Results Summary:")print(f"   Total test cases: {len(results)}")print(f"   Successful pipelines: {metrics['successful_pipelines']} ({metrics['pipeline_success_rate']:.1f}%)")print(f"   Avg total latency: {metrics['total_latency_p50']:.1f}ms (P50)")print(f"   Targets met: {passed_count}/{total_count}")